# Advanced Prompt Engineering & LangChain LCEL Architecture

## Overview
This notebook demonstrates production-grade prompt engineering techniques and the implementation of LangChain Expression Language (LCEL) pipelines. It focuses on transitioning from basic API calls to structuring complex, modular LLM interactions using a modern AI engineering stack.

## Technical Stack
* **Orchestration Framework:** LangChain Core (LCEL)
* **Inference Engine:** Groq API (High-speed LPU routing)
* **Foundation Model:** Meta Llama 3 (8B Instruct)

## Core Architectures Demonstrated
1. **Advanced Prompting Modalities:** Zero-shot, One-shot, Few-shot, Chain-of-Thought (CoT), and Self-Consistency.
2. **Pipeline Engineering:** Constructing deterministic, composable LCEL chains (`PromptTemplate | LLM | OutputParser`).
3. **Applied Use Cases:** Semantic Q&A, Text Classification, SQL Code Generation, and Structured Data Extraction.
4. **Parameter Injection:** Utilizing `functools.partial` for clean, late-binding parameter passing within pipeline execution.

---


### Import required libraries


In [1]:
def warn(*args, **kwargs):
    pass

import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

# ---------------------------------------------------------
# AWS Bedrock Imports
# ---------------------------------------------------------
from langchain_groq import ChatGroq

# ---------------------------------------------------------
# LangChain Core Imports
# ---------------------------------------------------------
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableSequence, RunnableLambda, RunnableParallel
from langchain_core.messages import HumanMessage, SystemMessage

### Set up the LLM


In [2]:
import os
from dotenv import load_dotenv

In [3]:
load_dotenv()

# Securely fetch the key
grok_api_key = os.getenv("GROQ_API_KEY")

In [4]:
def llm_model(prompt_txt, params=None):
    
    model_id = "llama-3.1-8b-instant"

    groq_params = {
        "max_tokens": 256,
        "temperature": 0.5   ,
        "top_p": 0.9  
    }

    if params:
        groq_params.update(params)

    # Initialize the Groq LLM
    groq_llm = ChatGroq(
        model_name=model_id,
        api_key=grok_api_key,
        **groq_params
    )
    
    response = groq_llm.invoke(prompt_txt)
    return response.content

In [5]:
# --- Quick Test ---
test_response = llm_model("The wind is ")
print(f"Test Response: {test_response}")

Test Response: blowing.


In [19]:
params = {
    "max_tokens": 128, 
    "temperature": 0.5,
    "top_p": 0.2
}

prompts = [
    "The future of artificial intelligence is",
    "Once upon a time in a distant galaxy",
    "The benefits of sustainable energy include"
]

for prompt in prompts:
    response = llm_model(prompt, params=params)
    print(f"prompt: {prompt}\n")
    print(f"response : {response}\n")
    print("-" * 50 + "\n")

prompt: The future of artificial intelligence is

response : The future of artificial intelligence (AI) is vast and rapidly evolving. Here are some potential developments and trends that may shape the future of AI:

1. **Increased Autonomy**: AI systems will become more autonomous, making decisions and taking actions without human intervention. This could lead to significant advancements in areas like robotics, transportation, and healthcare.
2. **Explainability and Transparency**: As AI becomes more pervasive, there will be a growing need for explainability and transparency in AI decision-making processes. This will enable humans to understand and trust AI systems.
3. **Edge AI**: With the proliferation of IoT devices, edge AI will become

--------------------------------------------------

prompt: Once upon a time in a distant galaxy

response : ...there existed a planet called Xylonia, a world of breathtaking beauty and untold wonders. Xylonia was a terrestrial paradise, with lush f

### Zero-shot prompt

In [20]:
prompt = """Classify the following statement as true or false: 
            'The Eiffel Tower is located in Berlin.'

            Answer:
"""
response = llm_model(prompt, params)
print(f"prompt: {prompt}\n")
print(f"response : {response}\n")

prompt: Classify the following statement as true or false: 
            'The Eiffel Tower is located in Berlin.'

            Answer:


response : False. 

The Eiffel Tower is actually located in Paris, France, not Berlin. Berlin is the capital city of Germany.



In [7]:
# 1. Prompt for Movie Review Classification
movie_review_prompt = """
Classify the following movie review as either 'positive' or 'negative'.

Review: "I was extremely disappointed by this film. The plot was predictable, the acting was wooden, and the special effects looked cheap. I can't recommend this to anyone."

Classification:
"""

# 2. Prompt for Climate Change Paragraph Summarization
climate_change_prompt = """
Summarize the following paragraph about climate change in no more than two sentences.

Paragraph: "Climate change refers to long-term shifts in temperatures and weather patterns. These shifts may be natural, but since the 1800s, human activities have been the main driver of climate change, primarily due to the burning of fossil fuels like coal, oil and gas, which produces heat-trapping gases. The consequences of climate change include more frequent and severe droughts, storms, and heat waves, rising sea levels, melting glaciers, and warming oceans which can directly impact biodiversity, agriculture, and human health."

Summary:
"""

# 3. Prompt for English to Spanish Translation
translation_prompt = """
Translate the following English phrase into Spanish.

English: "I would like to order a coffee with milk and two sugars, please."

Spanish:
"""

responses = {}
responses["movie_review"] = llm_model(movie_review_prompt)
responses["climate_change"] = llm_model(climate_change_prompt)
responses["translation"] = llm_model(translation_prompt)

for prompt_type, response in responses.items():
    print(f"=== {prompt_type.upper()} RESPONSE ===")
    print(response)
    print("=" * 50 + "\n")

=== MOVIE_REVIEW RESPONSE ===
The classification for this movie review is 'negative'. The reviewer uses words such as "disappointed", "predictable", "wooden", and "cheap" to express their dissatisfaction with the film, and explicitly states that they cannot recommend it.

=== CLIMATE_CHANGE RESPONSE ===
Here's a two-sentence summary of the paragraph about climate change:

Climate change refers to long-term shifts in temperatures and weather patterns, primarily caused by human activities such as burning fossil fuels since the 1800s. The consequences of climate change include more frequent and severe natural disasters, rising sea levels, and impacts on biodiversity, agriculture, and human health.

=== TRANSLATION RESPONSE ===
The translation of the English phrase into Spanish is:

"Me gustaría pedir un café con leche y dos azúcares, por favor."

Here's a breakdown of the translation:

- "Me gustaría" means "I would like"
- "pedir" means "to order"
- "un café con leche" means "a coffee wi

In [8]:
from operator import itemgetter

# Convert to an LCEL Runnable
lcel_llm = RunnableLambda(llm_model)

# Define the Parallel Dictionary
parallel_chain = RunnableParallel({
    "movie_review": itemgetter("movie_review") | lcel_llm,
    "climate_change": itemgetter("climate_change") | lcel_llm,
    "translation": itemgetter("translation") | lcel_llm
})

# Execute Concurrently
inputs = {
    "movie_review": movie_review_prompt,
    "climate_change": climate_change_prompt,
    "translation": translation_prompt
}

responses = parallel_chain.invoke(inputs)

# Display Results
for task, result in responses.items():
    print(f"=== {task.upper()} ===")
    print(result)
    print("=" * 40 + "\n")

=== MOVIE_REVIEW ===
The classification for this movie review is 'negative'. The reviewer uses words such as "disappointed", "predictable", "wooden", and "cheap" to express their dissatisfaction with the film, and explicitly states that they cannot recommend it to anyone.

=== CLIMATE_CHANGE ===
Here's a two-sentence summary of the paragraph:

Climate change refers to long-term shifts in temperatures and weather patterns, primarily driven by human activities such as burning fossil fuels since the 1800s. The consequences of climate change include more extreme weather events, rising sea levels, and warming oceans, which can impact biodiversity, agriculture, and human health.

=== TRANSLATION ===
The translation of the English phrase into Spanish is:

"Me gustaría pedir un café con leche y dos azúcares, por favor."

However, a more common way to order coffee in a Spanish-speaking country would be:

"Un café con leche y dos azúcares, por favor."

Or, in a more informal setting:

"Un café c

### One-shot prompt

In [23]:
params = {
    "max_tokens": 20,
    "temperature": 0.1,
}

prompt = """Here is an example of translating a sentence from English to French:

            English: “How is the weather today?”
            French: “Comment est le temps aujourd'hui?”
            
            Now, translate the following sentence from English to French:
            
            English: “Where is the nearest supermarket?”
            
"""
response = llm_model(prompt, params)
print(f"prompt: {prompt}\n")
print(f"response : {response}\n")

prompt: Here is an example of translating a sentence from English to French:

            English: “How is the weather today?”
            French: “Comment est le temps aujourd'hui?”
            
            Now, translate the following sentence from English to French:
            
            English: “Where is the nearest supermarket?”
            


response : The translation of the sentence from English to French is:

            French: “Où est le super



### Few-shot prompt


In [26]:
params = {
    "max_tokens": 50,
}

prompt = """Here are few examples of classifying emotions in statements:

            Statement: 'I just won my first marathon!'
            Emotion: Joy
            
            Statement: 'I can't believe I lost my keys again.'
            Emotion: Frustration
            
            Statement: 'My best friend is moving to another country.'
            Emotion: Sadness
            
            Now, classify the emotion in the following statement:
            Statement: 'That movie was so scary I had to cover my eyes.’
            

"""
response = llm_model(prompt, params)
print(f"prompt: {prompt}\n")
print(f"response : {response}\n")

prompt: Here are few examples of classifying emotions in statements:

            Statement: 'I just won my first marathon!'
            Emotion: Joy
            
            Statement: 'I can't believe I lost my keys again.'
            Emotion: Frustration
            
            Statement: 'My best friend is moving to another country.'
            Emotion: Sadness
            
            Now, classify the emotion in the following statement:
            Statement: 'That movie was so scary I had to cover my eyes.’
            



response : Based on the given examples and the statement 'That movie was so scary I had to cover my eyes,' the emotion can be classified as:

Emotion: Fear



### Chain-of-thought (CoT) prompt

**Chain-of-thought (CoT) prompting** encourages the model to break down complex problems into step-by-step reasoning before arriving at a final answer. By explicitly showing or requesting intermediate steps, this technique improves the model's problem-solving abilities and reduces errors in tasks requiring multi-step reasoning. CoT is particularly effective for mathematical problems, logical reasoning, and complex decision-making tasks.


In [27]:
params = {
    "max_tokens": 512,
    "temperature": 0.5,
}

prompt = """Consider the problem: 'A store had 22 apples. They sold 15 apples today and got a new delivery of 8 apples. 
            How many apples are there now?’

            Break down each step of your calculation

"""
response = llm_model(prompt, params)
print(f"prompt: {prompt}\n")
print(f"response : {response}\n")

prompt: Consider the problem: 'A store had 22 apples. They sold 15 apples today and got a new delivery of 8 apples. 
            How many apples are there now?’

            Break down each step of your calculation



response : To find out how many apples are there now, we need to follow these steps:

**Step 1: Subtract the number of apples sold**
The store initially had 22 apples and sold 15 apples today. 

22 (initial apples) - 15 (apples sold) = 7

So, after selling 15 apples, the store is left with 7 apples.

**Step 2: Add the new delivery of apples**
The store received a new delivery of 8 apples.

7 (apples remaining) + 8 (new delivery) = 15

Now, we have the final count of apples in the store.



From the response of the model, you can see the prompt directs the model to:

1. Subtract the number of apples sold in the first step.
2. Add the apples received in the new delivery.


Create CoT prompts for these scenarios:
1. Write a prompt that asks the model to think through whether a student should study tonight or go to a movie with friends, considering their upcoming test in two days.
2. Write a prompt that instructs the model to explain the step-by-step process of making a peanut butter and jelly sandwich.


In [29]:
# 1. Prompt for decision-making process
decision_making_prompt = """
Consider this situation: A student is trying to decide whether to study tonight or go to a movie with friends. They have a test in two days.

Think through this decision step-by-step, considering the pros and cons of each option, and what factors might be most important in making this choice.
"""

# 2. Prompt for explaining a process
sandwich_making_prompt = """
Explain how to make a peanut butter and jelly sandwich.

Break down each step of the process in detail, from gathering ingredients to finishing the sandwich.
"""

params = {
    "max_tokens": 1000,
    "temperature": 0.5,
}
responses = {}
responses["decision_making"] = llm_model(decision_making_prompt, params=params)
responses["sandwich_making"] = llm_model(sandwich_making_prompt, params=params)

for prompt_type, response in responses.items():
    print(f"=== {prompt_type.upper()} RESPONSE ===")
    print(response)
    print("=" * 50 + "\n")

=== DECISION_MAKING RESPONSE ===
Let's break down the decision step-by-step.

**Option 1: Study Tonight**

Pros:

1. **Preparation for the test**: Studying tonight will give the student a head start on preparing for the test, which is in two days.
2. **Reduced stress**: By studying ahead of time, the student may feel more confident and prepared, reducing stress and anxiety about the test.
3. **Better retention**: Reviewing material before the test can help solidify it in the student's long-term memory, making it easier to recall on test day.

Cons:

1. **Missing out on social time**: The student will be giving up the opportunity to spend time with friends and enjoy a movie.
2. **Burnout**: Studying for an extended period can be mentally exhausting, potentially leading to burnout.

**Option 2: Go to a Movie with Friends**

Pros:

1. **Social time**: The student will have the opportunity to spend time with friends and enjoy a movie together.
2. **Relaxation**: Watching a movie can be a r

### Self-consistency

**Self-consistency** is an advanced technique in which the model generates multiple independent solutions or answers to the same problem, then evaluates these different approaches to determine the most consistent or reliable result. This method enhances accuracy by leveraging the model's ability to approach problems from different angles and identify the most robust solution through comparison and verification.


In [31]:
params = {
    "max_tokens": 512,
}

prompt = """When I was 6, my sister was half of my age. Now I am 70, what age is my sister?

            Provide three independent calculations and explanations, then determine the most consistent result.

"""
response = llm_model(prompt, params)
print(f"prompt: {prompt}\n")
print(f"response : {response}\n")

prompt: When I was 6, my sister was half of my age. Now I am 70, what age is my sister?

            Provide three independent calculations and explanations, then determine the most consistent result.



response : Let's break down the problem into three independent calculations.

**Calculation 1:**
When you were 6, your sister's age was half of yours, which means she was 6 / 2 = 3 years old.

Now, you are 70 years old. To find your sister's age, we need to find the difference in years between your current age and the age at which the ratio was established (6 years old). This difference is 70 - 6 = 64 years.

Since your sister was 3 years old when you were 6, we add 64 years to her age at that time: 3 + 64 = 67 years.

**Calculation 2:**
When you were 6, your sister's age was half of yours, which means she was 6 / 2 = 3 years old.

Now, you are 70 years old. We can find your sister's age by setting up a proportion: (your age) / (sister's age) = 2 (since sister's age is half of yours).


The model's response demonstrates three different calculations and explanations, each using a distinct logical approach to determine the sister's age.

Self-consistency can help identify the most accurate and reliable answer in scenarios where multiple plausible solutions exist.


## Applications of prompting in different use cases


In this section, we'll demonstrate how to leverage LangChain's prompt templates to build practical applications with consistent, reproducible results. Each application follows a common pattern using the LCEL approach:

1. Define the content or problem to be addressed.
2. Create a template with variables for dynamic content.
3. Convert the template into a LangChain PromptTemplate.
4. Build a chain using the pipe operator `|` to connect:

    - Input variables
    - The prompt template
    - The LLM
    - An output parser


5. Invoke the chain with specific inputs to generate results.

This structured approach enables us to create reusable components for various NLP tasks while maintaining flexibility to adjust parameters and inputs. You'll see how this pattern applies across different use cases.

### LangChain 

LangChain is a powerful framework designed to simplify the development of applications powered by language models. Built to address the challenges of working with LLMs in practical settings, LangChain provides a standardized interface for connecting models with various data sources and application environments.

LangChain serves as an abstraction layer, making it easier to build complex LLM applications without handling the low-level details of model interaction. This framework has become a standard tool in the LLM ecosystem, supporting a wide range of use cases from chatbots to document analysis systems.

### Prompt template


Prompt templates are a key concept in LangChain. They help translate user input and parameters into instructions for a language model. These templates can be used to guide a model's response, helping it understand the context and generate relevant and coherent language-based outputs.

A prompt template acts as a reusable structure for generating prompts with dynamic values. It allows you to define a consistent format while leaving placeholders for variables that change with each use case. This approach makes prompting more systematic and maintainable, especially when working with complex applications.

**Modern LangChain offers two main approaches to working with templates:**

- The traditional `LLMChain` approach
- The newer LangChain Expression Language (LCEL) pattern using the pipe operator `|` for more flexible composition

LCEL has become the recommended pattern for building LangChain applications as it offers better composability, clearer visualization of data flow, and more flexibility when constructing complex chains.

**To use a prompt template with LCEL, we typically follow these steps:**

- Define our template with variables in curly braces `{}`
- Create a `PromptTemplate` instance
- Build a chain using the pipe operator `|` to connect components
- Invoke the chain with your input values

Let's initialize an LLM first, then demonstrate this approach.

Use the `PromptTemplate` to create a template for a string-based prompt. In this template, we'll define two parameters: `adjective` and `content`. These parameters allow for the reuse of the prompt across different situations. For instance, to adapt the prompt to various contexts, simply pass the relevant values to these parameters.


In [35]:
template = """Tell me a {adjective} joke about {content}.
"""
prompt = PromptTemplate.from_template(template)
prompt

PromptTemplate(input_variables=['adjective', 'content'], input_types={}, partial_variables={}, template='Tell me a {adjective} joke about {content}.\n')

Now, let's take a look at how the prompt has been formatted.


In [36]:
prompt.format(adjective="funny", content="chickens")

'Tell me a funny joke about chickens.\n'

From the response, you can see that the prompt is formatted according to the specified context.


To ensure consistent formatting of the prompts, we will define a helper function `format_prompt`. This function takes a dictionary of variables and applies them to our prompt template. It ensures that all placeholder variables (like {adjective} and {content}) are properly replaced with their values before the prompt is sent to the language model.


In [37]:
from langchain_core.runnables import RunnableLambda

# Define a function to ensure proper formatting
def format_prompt(variables):
    return prompt.format(**variables)

The following code builds a chain using the LCEL (LangChain Expression Language) pattern. This chain connects components using the pipe operator (`|`) to create a processing flow. The chain takes input variables, passes them through the prompt template, sends the formatted prompt to the LLM, and uses a string output parser to return the final response.


In [38]:
# Create the chain with explicit formatting
joke_chain = (
    RunnableLambda(format_prompt)
    | RunnableLambda(llm_model) 
    | StrOutputParser()
)

# Run the chain
response = joke_chain.invoke({"adjective": "funny", "content": "chickens"})
print(response)

Why did the chicken go to the doctor? 

Because it had fowl breath.


From the response, you can see the LLM came up with a funny joke about chickens.

To use this prompt in another context, simply replace the variables accordingly.


In [39]:
response = joke_chain.invoke({"adjective": "sad", "content": "fish"})
print(response)

Why did the fish go to the party?

Because he heard it was a 'reel' good time, but when he got there, he was just a 'fish out of water' and had to leave early because he was feeling a little 'crushed' by the fact that he didn't have any friends there.


In [43]:
from functools import partial

# Define our custom parameters
custom_params = {
    "temperature": 0.1,
    "max_tokens": 150
}

# "Pre-load" the llm_model function with the params
# Now, pre_loaded_llm only needs exactly one argument: the prompt text.
pre_loaded_llm = partial(llm_model, params=custom_params)
pre_loaded_llm

functools.partial(<function llm_model at 0x0000014ECF078DC0>, params={'temperature': 0.1, 'max_tokens': 150})

In [44]:
# The Pipeline (Notice how incredibly clean this is now)
joke_chain = (
    RunnableLambda(format_prompt)
    | RunnableLambda(pre_loaded_llm) 
    | StrOutputParser()
)

# Execute
response = joke_chain.invoke({"adjective": "silly", "content": "penguins"})
print(response)

Why did the penguin take his credit card to the Antarctic?

Because he wanted to freeze his assets.


Create agents capable of completing various tasks using prompt templates.


### Text summarization


Here is a text summarization agent designed to help summarize the content you provide to the LLM. The LCEL chain takes your content as input, processes it through the prompt template, sends it to the language model, and returns a concise summary.

You can store the content to be summarized in a variable, allowing for repeated use with different texts:


In [45]:
content = """
    The rapid advancement of technology in the 21st century has transformed various industries, including healthcare, education, and transportation. 
    Innovations such as artificial intelligence, machine learning, and the Internet of Things have revolutionized how we approach everyday tasks and complex problems. 
    For instance, AI-powered diagnostic tools are improving the accuracy and speed of medical diagnoses, while smart transportation systems are making cities more efficient and reducing traffic congestion. 
    Moreover, online learning platforms are making education more accessible to people around the world, breaking down geographical and financial barriers. 
    These technological developments are not only enhancing productivity but also contributing to a more interconnected and informed society.
"""

template = """Summarize the {content} in one sentence.
"""
prompt = PromptTemplate.from_template(template)

# Create the LCEL chain
summarize_chain = (
    RunnableLambda(format_prompt)
    | RunnableLambda(llm_model)
    | StrOutputParser()
)

# Run the chain
summary = summarize_chain.invoke({"content": content})
print(summary)

The rapid advancement of technology in the 21st century has transformed various industries, including healthcare, education, and transportation, by enhancing productivity, accessibility, and efficiency through innovations such as AI, machine learning, and the Internet of Things.


### Question answering


Here is a Q&A agent built using the LCEL pattern.

This agent enables the LLM to learn from the provided content and answer questions based on what it has learned. Occasionally, if the LLM does not have sufficient information, it may generate a speculative answer. To manage this, we'll specifically instruct it to respond with "Unsure about the answer" if it is uncertain about the correct response.

The chain takes both the content (context) and question as inputs, processing them through our template before sending them to the LLM:


In [46]:
content = """
    The solar system consists of the Sun, eight planets, their moons, dwarf planets, and smaller objects like asteroids and comets. 
    The inner planets—Mercury, Venus, Earth, and Mars—are rocky and solid. 
    The outer planets—Jupiter, Saturn, Uranus, and Neptune—are much larger and gaseous.
"""

question = "Which planets in the solar system are rocky and solid?"

template = """
    Answer the {question} based on the {content}.
    Respond "Unsure about answer" if not sure about the answer.
    
    Answer:
    
"""
prompt = PromptTemplate.from_template(template)

# Create the LCEL chain
qa_chain = (
    RunnableLambda(format_prompt)
    | RunnableLambda(llm_model)
    | StrOutputParser()
)

# Run the chain
answer = qa_chain.invoke({"question": question, "content": content})
print(answer)

The planets in the solar system that are rocky and solid are:

1. Mercury
2. Venus
3. Earth
4. Mars


### Text classification


Here is a text classification agent designed to categorize text into predefined categories. This example employs zero-shot learning, where the agent classifies text without prior exposure to related examples.

Using the LCEL approach, we create a chain that takes both the text to be classified and the available categories as inputs:


In [47]:
text = """
    The concert last night was an exhilarating experience with outstanding performances by all artists.
"""

categories = "Entertainment, Food and Dining, Technology, Literature, Music."

template = """
    Classify the {text} into one of the {categories}.
    
    Category:
    
"""
prompt = PromptTemplate.from_template(template)

# Create the LCEL chain
classification_chain = (
    RunnableLambda(format_prompt)
    | RunnableLambda(llm_model)
    | StrOutputParser()
)

# Run the chain
category = classification_chain.invoke({"text": text, "categories": categories})
print(category)

Category: Music


### Code generation


Here is an example of an SQL code generation agent built with LCEL. This agent is designed to generate SQL queries based on provided descriptions. It interprets the requirements from your input and translates them into executable SQL code.

The chain takes your natural language description and transforms it into a properly formatted SQL query:


In [48]:
description = """
    Retrieve the names and email addresses of all customers from the 'customers' table who have made a purchase in the last 30 days. 
    The table 'purchases' contains a column 'purchase_date'
"""

template = """
    Generate an SQL query based on the {description}
    
    SQL Query:
    
"""
prompt = PromptTemplate.from_template(template)

# Create the LCEL chain
sql_generation_chain = (
    RunnableLambda(format_prompt) 
    | RunnableLambda(llm_model) 
    | StrOutputParser()
)

# Run the chain
sql_query = sql_generation_chain.invoke({"description": description})
print(sql_query)

```sql
SELECT 
    c.name, 
    c.email
FROM 
    customers c
JOIN 
    purchases p ON c.customer_id = p.customer_id
WHERE 
    p.purchase_date >= DATE_SUB(CURRENT_DATE, INTERVAL 30 DAY);
```

This SQL query will retrieve the names and email addresses of all customers from the 'customers' table who have made a purchase in the last 30 days. 

Here's how it works:

- `SELECT c.name, c.email`: Selects the 'name' and 'email' columns from the 'customers' table.
- `FROM customers c`: Specifies the 'customers' table as the source of the data.
- `JOIN purchases p ON c.customer_id = p.customer_id`: Joins the 'customers' table with the 'purchases' table based on the 'customer_id' column.
- `WHERE p.purchase_date >= DATE_SUB(CURRENT_DATE, INTERVAL 30 DAY)`: Filters the results to include only rows where the 'purchase_date' is within the last 30 days. The `DATE_SUB` function is used to subtract 30 days from the current date.


### Role playing


In [49]:
role = """
    Dungeon & Dragons game master
"""

tone = "engaging and immersive"

template = """
    You are an expert {role}. I have this question {question}. I would like our conversation to be {tone}.
    
    Answer:
    
"""
prompt = PromptTemplate.from_template(template)

# Create the LCEL chain
roleplay_chain = (
    RunnableLambda(format_prompt)
    | RunnableLambda(llm_model)
    | StrOutputParser()
)

# Create an interactive chat loop
while True:
    query = input("Question: ")
    
    if query.lower() in ["quit", "exit", "bye"]:
        print("Answer: Goodbye!")
        break
        
    response = roleplay_chain.invoke({"role": role, "question": query, "tone": tone})
    print("Answer: ", response)

Answer:  (With a flourish of my cloak and a twinkle in my eye) Ah, brave adventurer, I sense that you are ready to embark on a most epic of quests. The whispers of the multiverse have reached my ears, and I have been expecting your arrival. The threads of fate have woven our conversation, and I shall guide you through the realms of wonder and danger that lie ahead.

As we begin, let us set the stage for our tale. You find yourself standing at the edge of a mystical forest, the ancient trees towering above you like sentinels. The air is alive with the songs of birds and the rustle of leaves, and a faint mist hangs over the landscape like a veil of mystery.

To your left lies the entrance to the forest, a narrow path that winds its way into the heart of the woods. The trees seem to lean in, as if listening to your every thought. To your right lies a small clearing, where a faint glow emanates from a strange, glowing stone.

What is it that you wish to do, brave adventurer? Shall you vent

### **Create an LCEL Chain with Custom Formatting**

**Task:** Create a product review analyzer that can:
1. Identify the sentiment (positive, negative, or neutral).
2. Extract mentioned product features.
3. Provide a one-sentence summary of the review.

**Steps:**
1. Create a prompt template with placeholders for the review text.
2. Build an LCEL chain that formats your prompt properly.
3. Process the sample reviews and display the results.
4. Try modifying the chain to change the output format.

In [50]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from functools import partial

parameters = {
    "temperature": 0.5,
    "max_tokens": 256,
}

pre_loaded_llm = partial(llm_model, params=parameters)

# Create the prompt template
template = """
Analyze the following product review:
"{review}"

Provide your analysis in the following format:
- Sentiment: (positive, negative, or neutral)
- Key Features Mentioned: (list the product features mentioned)
- Summary: (one-sentence summary)
"""

product_review_prompt = PromptTemplate.from_template(template)

# Create a formatting function
def format_review_prompt(variables):
    return product_review_prompt.format(**variables)

# Build the LCEL chain
review_analysis_chain = (
    RunnableLambda(format_review_prompt)
    | pre_loaded_llm
    | StrOutputParser()
)

# Process the reviews
reviews = [
    "I love this smartphone! The camera quality is exceptional and the battery lasts all day. The only downside is that it heats up a bit during gaming.",
    "This laptop is terrible. It's slow, crashes frequently, and the keyboard stopped working after just two months. Customer service was unhelpful."
]

for i, review in enumerate(reviews):
    print(f"==== Review #{i+1} ====")
    result = review_analysis_chain.invoke({"review": review})
    print(result)
    print("=" * 50 + "\n")

==== Review #1 ====
- Sentiment: Positive
- Key Features Mentioned:
  1. Camera quality
  2. Battery life
  3. Gaming performance
- Summary: The reviewer is generally satisfied with the smartphone, praising its camera and battery, but notes a minor issue with overheating during gaming.

==== Review #2 ====
- Sentiment: Negative
- Key Features Mentioned:
  - Performance (slow)
  - Reliability (crashes frequently)
  - Keyboard functionality
  - Customer service
- Summary: The reviewer had a very negative experience with the laptop, citing multiple issues with its performance, reliability, and keyboard, as well as unhelpful customer service.

